# Range Pricer

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('generated')))

import numpy as np
from scipy.stats import norm
import flatbuffers
import ipywidgets as w
from IPython.display import display
import RangePricer.PricingRequest as PR

## Pricing function

In [ ]:
def price_range(spot, low, high, vol, rate, expiry):
    sqrt_t = np.sqrt(expiry)
    drift  = (rate - 0.5 * vol**2) * expiry
    d_low  = (np.log(spot / low)  + drift) / (vol * sqrt_t)
    d_high = (np.log(spot / high) + drift) / (vol * sqrt_t)
    return np.exp(-rate * expiry) * (norm.cdf(d_low) - norm.cdf(d_high))

## FlatBuffers helpers

In [ ]:
def build_request(request_id, spot, low, high, vol, rate, expiry):
    """Serialise parameters into a PricingRequest FlatBuffer, return bytes."""
    builder = flatbuffers.Builder(128)
    rid = builder.CreateString(request_id)
    PR.Start(builder)
    PR.AddRequestId(builder, rid)
    PR.AddSpot(builder, spot)
    PR.AddLow(builder, low)
    PR.AddHigh(builder, high)
    PR.AddVol(builder, vol)
    PR.AddRate(builder, rate)
    PR.AddExpiry(builder, expiry)
    root = PR.End(builder)
    builder.Finish(root)
    return bytes(builder.Output())

def read_request(buf):
    """Deserialise bytes back into a PricingRequest object."""
    return PR.PricingRequest.GetRootAs(buf, 0)

## Pricing request UI

In [ ]:
# ── widgets ────────────────────────────────────────────────────────────────
w_id     = w.Text(value='req-001',  description='Request ID', style={'description_width': '100px'})
w_spot   = w.FloatText(value=100.0, description='Spot',       style={'description_width': '100px'})
w_low    = w.FloatText(value=95.0,  description='Low',        style={'description_width': '100px'})
w_high   = w.FloatText(value=105.0, description='High',       style={'description_width': '100px'})
w_vol    = w.FloatSlider(value=0.20, min=0.01, max=1.0, step=0.01,
                         description='Vol',    readout_format='.2f',
                         style={'description_width': '100px'})
w_rate   = w.FloatSlider(value=0.05, min=0.0,  max=0.20, step=0.005,
                         description='Rate',   readout_format='.3f',
                         style={'description_width': '100px'})
w_expiry = w.FloatSlider(value=1.0,  min=0.01, max=5.0,  step=0.01,
                         description='Expiry', readout_format='.2f',
                         style={'description_width': '100px'})

btn    = w.Button(description='Price', button_style='primary')
out    = w.Output()

# ── callback ───────────────────────────────────────────────────────────────
def on_price(_):
    out.clear_output(wait=True)
    with out:
        buf = build_request(
            w_id.value, w_spot.value, w_low.value,
            w_high.value, w_vol.value, w_rate.value, w_expiry.value
        )
        req = read_request(buf)
        pv  = price_range(req.Spot(), req.Low(), req.High(),
                          req.Vol(), req.Rate(), req.Expiry())
        print(f'Request : {req.RequestId().decode()}')
        print(f'Spot    : {req.Spot():.4f}')
        print(f'Range   : [{req.Low():.4f}, {req.High():.4f}]')
        print(f'Vol     : {req.Vol():.4f}   Rate: {req.Rate():.4f}   Expiry: {req.Expiry():.4f}')
        print(f'Buffer  : {len(buf)} bytes')
        print(f'──────────────────────────────')
        print(f'PV      : {pv:.6f}')

btn.on_click(on_price)

display(w.VBox([
    w.HTML('<b>Pricing Request</b>'),
    w_id, w_spot, w_low, w_high, w_vol, w_rate, w_expiry,
    btn, out
]))